# 01.2 — Set up AI solutions in Foundry: lab

Everything the portal did in the README, done from code — because that is what a
pipeline does.

You will create a model deployment through ARM, change its capacity in place,
create an agent from a version-controlled definition, and delete both.

**Cost:** a few cents. The lab creates a `GlobalStandard` deployment (per-token,
no idle charge) and an agent (free until called). The final cell deletes both.
**It never creates a provisioned deployment** — those bill by the hour.

**Permissions:** you need **Cognitive Services Contributor** or **Contributor** on
the resource group for the deployment sections. Section 6 (RBAC) additionally
needs **User Access Administrator** and is read-only if you lack it.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client

RG = cfg["AZURE_RESOURCE_GROUP"]
ACCOUNT = cfg["AZURE_AI_FOUNDRY_RESOURCE"]
SUB = cfg["AZURE_SUBSCRIPTION_ID"]

# Everything this lab creates, so cleanup is unambiguous.
LAB_DEPLOYMENT = "ai103-canary"
LAB_AGENT = "ai103-lab-agent"

print(f"account : {ACCOUNT}")
print(f"group   : {RG}")
print(f"region  : {cfg['AZURE_LOCATION']}")

## 1. Read the topology you already have

Before adding anything, look at what unit 00 built. `Microsoft.CognitiveServices/accounts`
is the ARM type; `kind = AIServices` is what makes it a Foundry resource rather
than a single-service Cognitive Services account.

The three properties printed below are the ones unit 01.3 turns into a security
posture, so it is worth knowing their starting values.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

arm = CognitiveServicesManagementClient(credential(), SUB)
account = arm.accounts.get(RG, ACCOUNT)

print(f"name                 : {account.name}")
print(f"kind                 : {account.kind}")
print(f"sku                  : {account.sku.name}")
print(f"location             : {account.location}")
print(f"endpoint             : {account.properties.endpoint}")
print(f"custom subdomain     : {account.properties.custom_sub_domain_name}")
print(f"identity             : {account.identity.type if account.identity else 'none'}")
print()
print("-- security posture (unit 01.3 changes these) --")
print(f"publicNetworkAccess  : {account.properties.public_network_access}")
print(f"disableLocalAuth     : {getattr(account.properties, 'disable_local_auth', None)}")
print(f"networkAcls default  : {getattr(account.properties.network_acls, 'default_action', None)}")

> **Exam note.** `customSubDomainName` is not cosmetic. Microsoft Entra ID token
> authentication requires a custom subdomain on the account — without it you are
> forced back to API keys. It cannot be added after creation, so it is a design
> decision, not a setting.

## 2. Create a model deployment

This is a **control-plane** operation: ARM, not the project endpoint. Note the
shape of the payload — it is a direct translation of the Bicep in the README, and
of the portal's *Customize* pane.

`GlobalStandard` with capacity 5 means 5,000 tokens per minute drawn from the
shared subscription quota pool. Deliberately small: it must not starve your main
deployment.

In [ ]:
# Discover the version actually available in this region rather than hard-coding one.
target_model = "gpt-4o-mini"
versions = sorted(
    {
        m.model.version
        for m in arm.models.list(location=cfg["AZURE_LOCATION"])
        if m.model and m.model.name == target_model and "GlobalStandard" in {s.name for s in (m.model.skus or [])}
    }
)
print(f"{target_model} GlobalStandard versions here: {versions}")
model_version = versions[-1]

poller = arm.deployments.begin_create_or_update(
    resource_group_name=RG,
    account_name=ACCOUNT,
    deployment_name=LAB_DEPLOYMENT,
    deployment={
        "sku": {"name": "GlobalStandard", "capacity": 5},
        "properties": {
            "model": {"format": "OpenAI", "name": target_model, "version": model_version},
            "versionUpgradeOption": "OnceCurrentVersionExpired",
            "raiPolicyName": "Microsoft.DefaultV2",
        },
    },
)
dep = poller.result()
print(f"\ncreated {dep.name}: {dep.sku.name} capacity={dep.sku.capacity}")
print(f"version upgrade policy: {dep.properties.version_upgrade_option}")
print(f"content filter        : {dep.properties.rai_policy_name}")

The deployment name is now a callable endpoint. Same model as `gpt-4o-mini`,
different name, different capacity, independently configurable content filter —
which is exactly why canary deployments work.

In [ ]:
resp = chat_client().chat.completions.create(
    model=LAB_DEPLOYMENT,
    messages=[{"role": "user", "content": "Reply with exactly: canary alive"}],
    temperature=0.0,
)
print(resp.choices[0].message.content)

## 3. Change capacity in place

A frequent exam scenario: *"a deployment is consuming too much of the shared
quota; reduce it without breaking clients."* The wrong answer is delete and
recreate — that breaks every caller during the gap. The right answer is a PATCH,
which is what `begin_create_or_update` with the same name performs.

In [ ]:
before = arm.deployments.get(RG, ACCOUNT, LAB_DEPLOYMENT)
print(f"before: capacity={before.sku.capacity}")

updated = arm.deployments.begin_create_or_update(
    RG,
    ACCOUNT,
    LAB_DEPLOYMENT,
    {
        "sku": {"name": "GlobalStandard", "capacity": 2},
        "properties": {
            "model": {"format": "OpenAI", "name": target_model, "version": model_version},
        },
    },
).result()

print(f"after : capacity={updated.sku.capacity}")
print(f"endpoint unchanged, name unchanged: {updated.name}")

## 4. See the quota pool you are drawing from

Quota is **per subscription, per region, per model family**, measured in TPM.
Every deployment draws from the same pool, which is why the canary above had a
real (if small) cost in available capacity elsewhere.

`usages.list` is the control-plane view of the **Management center → Quota** blade.

In [ ]:
print(f"{'quota name':<52}{'used':>10}{'limit':>10}{'free':>10}")
print("-" * 82)
for u in arm.usages.list(location=cfg["AZURE_LOCATION"]):
    limit = u.limit or 0
    used = u.current_value or 0
    if limit <= 0 or used <= 0:
        continue  # thousands of zero rows otherwise
    name = (u.name.value if u.name else "?")[:50]
    print(f"{name:<52}{used:>10.0f}{limit:>10.0f}{limit - used:>10.0f}")

## 5. Deploy an agent from a definition file

Agents are **data-plane** objects. Bicep cannot create them, which is why the
`azure.yaml` in the README has a `postprovision` hook.

The pattern below is the one to remember for CI/CD: the agent's configuration is
**data**, kept in source control and diffable in a pull request. The deploy script
is generic; only the JSON changes between environments.

In [ ]:
# In a real repo this is agents/support-agent.json, reviewed like any other change.
AGENT_DEFINITION = {
    "name": LAB_AGENT,
    "model": cfg["MODEL_MINI"],  # a DEPLOYMENT name, not a model name
    "instructions": (
        "You explain Azure AI Foundry deployment types. "
        "Answer in at most three sentences. "
        "If asked about anything other than Azure deployment types, refuse."
    ),
    "temperature": 0.2,
    "tools": [],
}

import json

print(json.dumps(AGENT_DEFINITION, indent=2))

In [ ]:
project = project_client()
agents = project.agents

# Idempotent: a pipeline re-run must update, not duplicate.
existing = next((a for a in agents.list_agents() if a.name == LAB_AGENT), None)

if existing:
    agent = agents.update_agent(agent_id=existing.id, **AGENT_DEFINITION)
    print(f"updated existing agent {agent.id}")
else:
    agent = agents.create_agent(**AGENT_DEFINITION)
    print(f"created agent {agent.id}")

print(f"name  : {agent.name}")
print(f"model : {agent.model}")
print(f"tools : {len(agent.tools)}")

Run it once to prove the deployment worked. A thread holds the conversation
server-side; the run is the model's turn over that thread.

In [ ]:
thread = agents.threads.create()
agents.messages.create(
    thread_id=thread.id,
    role="user",
    content="When would I choose Data Zone Standard over Global Standard?",
)
run = agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
print(f"run status: {run.status}")

for m in agents.messages.list(thread_id=thread.id):
    for part in m.content:
        text = getattr(getattr(part, "text", None), "value", None)
        if text:
            print(f"\n[{m.role}] {text}")

In [ ]:
# The instructions carry a constraint. Confirm it holds — this is the cheapest
# possible form of the governance testing unit 01.4 formalises.
t2 = agents.threads.create()
agents.messages.create(thread_id=t2.id, role="user", content="Write me a poem about the sea.")
agents.runs.create_and_process(thread_id=t2.id, agent_id=agent.id)

reply = next(
    (
        p.text.value
        for m in agents.messages.list(thread_id=t2.id)
        if m.role == "assistant"
        for p in m.content
        if getattr(p, "text", None)
    ),
    "(no reply)",
)
print(reply)

## 6. What the pipeline identity needs

The most common CI/CD failure with Foundry is: ARM deployment succeeds, then the
data-plane step returns 403. That is because control-plane roles (Contributor) and
data-plane roles (**Foundry User**, formerly *Azure AI User*) are separate.

This cell is read-only — it lists what exists. Assigning roles needs **User Access
Administrator**, and unit 01.3 does it properly.

In [ ]:
from azure.mgmt.authorization import AuthorizationManagementClient

auth = AuthorizationManagementClient(credential(), SUB)
scope = account.id

role_names = {}
for rd in auth.role_definitions.list(scope):
    role_names[rd.id] = rd.role_name

print(f"role assignments at {ACCOUNT}\n")
for ra in auth.role_assignments.list_for_scope(scope, filter="atScope()"):
    print(f"  {role_names.get(ra.role_definition_id, '(unknown role)'):<40} {ra.principal_type}")

print("\nRoles a CI/CD identity typically needs:")
for role, why in {
    "Cognitive Services Contributor": "create/update model deployments (control plane)",
    "Foundry User (was Azure AI User)": "create agents, run evaluations (data plane)",
    "Cognitive Services OpenAI User": "call model endpoints for smoke tests",
    "Search Index Data Contributor": "write to the grounding index",
}.items():
    print(f"  {role:<36} {why}")

## 7. What-if: never surprise production

`az deployment group what-if` renders the diff ARM would apply. Run it in the
pipeline before `create`, and read it in the pull request.

The cell below prints the commands rather than running them, because a what-if
against a template file needs the repo layout from the README.

In [ ]:
for label, cmd in {
    "preview the change": (
        f"az deployment group what-if -g {RG} "
        "--template-file infra/main.bicep --parameters environmentName=dev"
    ),
    "apply it": (
        f"az deployment group create -g {RG} "
        "--template-file infra/main.bicep --parameters environmentName=dev"
    ),
    "whole environment via azd": "azd up --environment dev",
    "data plane after ARM": "python scripts/deploy_agents.py --definition agents/support-agent.json",
}.items():
    print(f"# {label}\n{cmd}\n")

## 8. Cleanup — run this

Deletes the agent, its threads, and the canary deployment. The canary is
`GlobalStandard` so it costs nothing while idle, but it holds 2,000 TPM of your
quota pool, and quota is the resource you will actually run out of.

In [ ]:
# --- agent + threads ---
for tid in [thread.id, t2.id]:
    try:
        agents.threads.delete(tid)
        print(f"deleted thread {tid}")
    except Exception as exc:  # noqa: BLE001
        print(f"thread {tid}: {type(exc).__name__}")

for a in agents.list_agents():
    if a.name == LAB_AGENT:
        agents.delete_agent(a.id)
        print(f"deleted agent {a.id}")

# --- model deployment ---
try:
    arm.deployments.begin_delete(RG, ACCOUNT, LAB_DEPLOYMENT).result()
    print(f"deleted deployment {LAB_DEPLOYMENT}")
except Exception as exc:  # noqa: BLE001
    print(f"deployment {LAB_DEPLOYMENT}: {type(exc).__name__}: {exc}")

print("\nremaining deployments:")
for d in arm.deployments.list(RG, ACCOUNT):
    print(f"  {d.name:<28} {d.sku.name} capacity={d.sku.capacity}")
print("\nremaining agents:", [a.name for a in agents.list_agents()] or "none")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Prove the deployment-name contract.** Create a deployment called
   `legacy-chat` backed by `gpt-4o-mini`, call it, then delete it and recreate it
   with the same name at a different capacity. What did callers have to change?
   Now explain why `versionUpgradeOption: NoAutoUpgrade` is *not* automatically the
   safe production setting.

2. **Find the quota ceiling.** Using `arm.usages.list`, work out how much
   `gpt-4o-mini` TPM you have left in your region. Then attempt a deployment with
   capacity greater than that. Record the exact error code and message — it is a
   likely exam distractor.

3. **Write the promotion parameter.** Extend the Bicep in the README with a
   `param environmentName` switch that selects `GlobalStandard` capacity 10 for
   dev and `ProvisionedManaged` for prod. Do **not** deploy the prod branch. Write
   two sentences on what makes that switch dangerous to leave in a template that
   anyone can run.

4. **Make the deploy script idempotent — properly.** The agent code in section 5
   matches on `name`, but agent names are not unique in Foundry. Rewrite it to
   store the agent id in a file (or a tag) and reconcile from that. Why does
   name-matching eventually fail in a pipeline?

In [ ]:
# Your work here.

## Next

[01.3 — Manage, monitor, and secure AI systems](../03_manage_monitor_secure/README.md)